# Setup

In [3]:
!nvidia-smi
%pip install ultralytics
import ultralytics
ultralytics.checks()
import os
HOME = os.getcwd()
print("HOME:", HOME)
%pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
%pip install -q jupyter_bbox_widget roboflow dataclasses-json supervision
!mkdir -p {HOME}/weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -P {HOME}/weights
CHECKPOINT_PATH = os.path.join(HOME, "weights", "sam_vit_h_4b8939.pth")
print(CHECKPOINT_PATH, "; exist:", os.path.isfile(CHECKPOINT_PATH))
import torch
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_TYPE = "vit_h"

Ultralytics YOLOv8.2.50 🚀 Python-3.10.12 torch-2.3.0+cu121 CUDA:0 (Tesla T4, 15102MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 30.3/78.2 GB disk)
HOME: /content
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.2/76.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.0/124.0 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.7 MB/s eta 0:00:00
/content/weights/sam_vit_h_4b8939.pth ; exist: True


# Imports

In [4]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
from ultralytics import YOLO
import numpy as np

#YOLOv8 bounding box extraction

In [5]:
desired_label=["remote","bottle","cup","bowl","vase","scissors","toothbrush","wine glass","baseball bat"]

In [6]:
def process_image(IMAGE_PATH,desired_label):
  model = YOLO("yolov8n.pt")
  results = model([IMAGE_PATH])
  for result in results:
    boxes = result.boxes
    if(boxes.shape[0]==0):
      return [0,0,0,0]
    for i in range(boxes.shape[0]):
        if (model.names[int(boxes.cls[i].item())] in desired_label):
          bounding_box=np.array([(boxes.xyxy[i].cpu().numpy()[0]),
                        (boxes.xyxy[i].cpu().numpy()[1]),
                        (boxes.xyxy[i].cpu().numpy()[2]),
                        (boxes.xyxy[i].cpu().numpy()[3])])

          return bounding_box
        else:
          return [0,0,0,0]


#SAM segmentation

In [7]:
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)

In [8]:
mask_generator = SamAutomaticMaskGenerator(sam)

In [9]:
mask_predictor = SamPredictor(sam)

In [10]:
import numpy as np
import supervision as sv
def sam_segment(bbox,IMAGE_PATH):
  image_bgr = cv.imread(IMAGE_PATH)
  image_rgb = cv.cvtColor(image_bgr, cv.COLOR_BGR2RGB)
  mask_predictor.set_image(image_rgb)

  masks, scores, logits = mask_predictor.predict(
      box=bbox,
      multimask_output=False
  )
  detections = sv.Detections(
    xyxy=sv.mask_to_xyxy(masks=masks),
    mask=masks
  )
  box_annotator = sv.BoxAnnotator(color=sv.Color.red())
  mask_annotator = sv.MaskAnnotator(color=sv.Color.red(), color_lookup=sv.ColorLookup.INDEX)

  detections = sv.Detections(
      xyxy=sv.mask_to_xyxy(masks=masks),
      mask=masks
  )
  detections = detections[detections.area == np.max(detections.area)]

  source_image = box_annotator.annotate(scene=image_bgr.copy(), detections=detections, skip_label=True)
  segmented_image = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)
  return detections

In [20]:
import cv2 as cv
import random
from math import atan2, cos, sin, sqrt, pi , tan , radians
from shapely.geometry import LineString,Polygon
from google.colab.patches import cv2_imshow
from itertools import combinations
def calculate_angle_and_coordinates(ann,image,bbox):
      images=[]
      bw=ann.mask[0].astype(int)
      bw = np.array(bw, np.uint8)
      contours, _ = cv.findContours(bw, cv.RETR_LIST, cv.CHAIN_APPROX_NONE)
      points=[]
      x=random.randint(int(bbox[0]),int(bbox[2]))
      y=random.randint(int(bbox[1]),int(bbox[3]))
      x_limit=list(range(x-1024, x+1024))
      x_limit=np.array(x_limit)
      y_limit=[]
      for k in range(-1024,1024):
            y_limit.append(y)
      for k, c in enumerate(contours):
        c=np.reshape(c,(-1,2))
        line_1=LineString(np.column_stack((x_limit,y_limit)))
        if(len(c)<4):
          return None
        line_2=Polygon(c)
        intersection=line_1.intersection(line_2,grid_size=1)
        if type(intersection) is LineString:
          if intersection:
            points.append([intersection.xy[0][0],intersection.xy[1][0]])
            points.append([intersection.xy[0][-1],intersection.xy[1][-1]])
      points.sort()
      if points:
        image_copy=image.copy()
        cv.circle(image_copy,(int(points[0][0]),int(points[0][1])),20,(0,255,0),-1)
        cv.circle(image_copy,(int(points[1][0]),int(points[1][1])),20,(0,255,0),-1)
        images.append(image_copy)
      if (len(points)>2):
        image_copy=image.copy()
        cv.circle(image_copy,(int(points[-1][0]),int(points[-1][1])),20,(0,255,0),-1)
        cv.circle(image_copy,(int(points[-2][0]),int(points[-2][1])),20,(0,255,0),-1)
        images.append(image_copy)
      return images

In [21]:
input_folder = "/content/drive/MyDrive/LLM final project/image dataset"
output_folder = "/content/drive/MyDrive/LLM final project/augs"
for filename in os.listdir(input_folder):
  if filename != ".config":
    image_path = os.path.join(input_folder, filename)
    image_bgr = cv.imread(image_path)
    image_rgb = cv.cvtColor(image_bgr, cv.COLOR_BGR2RGB)
    bbox=process_image(image_path,desired_label)
    if(type(bbox)!=list):
      detections=sam_segment(bbox,image_path)
      for i in range(5):
        image = calculate_angle_and_coordinates(detections,image_rgb,bbox)
        if(image is not None):
          for j in range(len(image)):
            name = f'{i+j}_{filename}'
            output_path = os.path.join(output_folder, name)
            cv.imwrite(output_path, image[j])


0: 640x640 2 remotes, 8.8ms
Speed: 4.1ms preprocess, 8.8ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)


SupervisionWarnings: red is deprecated: `Color.red()` is deprecated and will be removed in `supervision-0.22.0`. Use `Color.RED` instead.
SupervisionWarnings: red is deprecated: `Color.red()` is deprecated and will be removed in `supervision-0.22.0`. Use `Color.RED` instead.
SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead



0: 640x640 1 wine glass, 6.0ms
Speed: 2.9ms preprocess, 6.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)



SupervisionWarnings: red is deprecated: `Color.red()` is deprecated and will be removed in `supervision-0.22.0`. Use `Color.RED` instead.
SupervisionWarnings: red is deprecated: `Color.red()` is deprecated and will be removed in `supervision-0.22.0`. Use `Color.RED` instead.
SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead


0: 640x640 (no detections), 6.0ms
Speed: 3.0ms preprocess, 6.0ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 vase, 5.9ms
Speed: 2.7ms preprocess, 5.9ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)


...


0: 640x640 (no detections), 5.9ms
Speed: 2.7ms preprocess, 5.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 cup, 6.0ms
Speed: 2.8ms preprocess, 6.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)


SupervisionWarnings: red is deprecated: `Color.red()` is deprecated and will be removed in `supervision-0.22.0`. Use `Color.RED` instead.
SupervisionWarnings: red is deprecated: `Color.red()` is deprecated and will be removed in `supervision-0.22.0`. Use `Color.RED` instead.
SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead



0: 640x640 1 vase, 5.9ms
Speed: 2.9ms preprocess, 5.9ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)


SupervisionWarnings: red is deprecated: `Color.red()` is deprecated and will be removed in `supervision-0.22.0`. Use `Color.RED` instead.
SupervisionWarnings: red is deprecated: `Color.red()` is deprecated and will be removed in `supervision-0.22.0`. Use `Color.RED` instead.
SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead



0: 640x640 (no detections), 6.0ms
Speed: 2.8ms preprocess, 6.0ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
